# Da She's Voice Engine — Colab GPU Accelerated
Run this to provide GPU-accelerated TTS as a fallback.
The voice server on Echad will detect this endpoint automatically.

In [ ]:
# 1. Install dependencies (no version pins — Colab's defaults work)
!pip install -q TTS flask flask-cors 2>&1 | tail -5
print("Dependencies installed")

In [ ]:
# 2. Download speaker voice sample
import requests
SPEAKER_URL = "https://qwert.crousia.com/speaker.wav"
resp = requests.get(SPEAKER_URL, timeout=30)
resp.raise_for_status()
with open("speaker.wav", "wb") as f:
    f.write(resp.content)
print(f"Downloaded speaker sample: {len(resp.content)} bytes")

In [ ]:
# 3. Load XTTS v2 on GPU
from TTS.api import TTS
import torch

has_gpu = torch.cuda.is_available()
print(f"CUDA: {has_gpu}  GPU: {torch.cuda.get_device_name(0) if has_gpu else 'N/A'}")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=has_gpu)
print("XTTS v2 ready")

In [ ]:
# 4. Start Flask server
from flask import Flask, request, send_file
import tempfile, os, threading

app = Flask(__name__)

@app.route("/health")
def health():
    return {"status": "ok", "gpu": has_gpu}

@app.route("/synthesize", methods=["POST"])
def synthesize():
    data = request.get_json()
    text = data.get("text", "") if data else ""
    if not text:
        return {"error": "text required"}, 400
    fd, path = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    tts.tts_to_file(text=text, file_path=path, speaker_wav="speaker.wav", language="en")
    return send_file(path, mimetype="audio/wav")

threading.Thread(target=lambda: app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False), daemon=True).start()
print("Server on port 5000")

In [ ]:
# 5. Expose via Cloudflare Tunnel
import subprocess, time, re, json

!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared 2>&1 && chmod +x /usr/local/bin/cloudflared

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# Wait up to 20 seconds for the URL
url = None
for _ in range(20):
    time.sleep(1)
    out = proc.stdout.read(2048) if proc.stdout else ""
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', out)
    if m:
        url = m.group(0)
        break

print("=" * 60)
if url:
    print(f"COLAB GPU ENDPOINT: {url}")
    print("=" * 60)
    print(f"Set this as COLAB_GPU_URL in Echad's environment for GPU fallback.")
else:
    print("Tunnel output:")
    print(out[-500:] if len(out) > 500 else out)

while True:
    time.sleep(60)